# 技巧: 保存和加载仿真结果

## 目的
运行一个复杂的、耗时的模拟后，我们通常不希望每次想查看或分析结果时都重新运行一遍。因此，将模拟结果保存到文件中，并在需要时重新加载，是一个非常重要和常见的操作。

当使用`SimulationController`并设置`log_history=True`时，返回的`history`对象是一个包含了嵌套字典和Numpy数组的复杂Python对象列表。对于这种复杂的对象，使用`pickle`模块进行序列化和反序列化是最直接有效的方法。

此笔记本将演示：
1.  运行一个简单的模拟并获取`history`对象。
2.  使用`pickle.dump()`将`history`对象保存到文件中。
3.  使用`pickle.load()`从文件中加载`history`对象。
4.  验证加载后的对象可以被正常使用。

## 1. 运行模拟并生成结果

我们首先运行一个简单的网络模拟，以生成一个`history`对象作为我们的数据源。

In [ ]:
import sys, os, pickle
import numpy as np
import matplotlib.pyplot as plt
sys.path.insert(0, os.path.abspath(os.path.join(os.path.dirname('__file__'), '..')))

from common.controller import SimulationController
from common.base_model import BaseModelComponent

# --- 准备一个简单的模拟 ---
class Source(BaseModelComponent):
    def __init__(self, name: str):
        super().__init__(name)
    def step(self, inflows, dt): self.outflow = np.random.rand() * 10
class Sink(BaseModelComponent):
    def __init__(self, name: str):
        super().__init__(name)
    def step(self, inflows, dt): self.outflow = sum(inflows.values())

controller = SimulationController()
controller.add_component(Source(name="S1"))
controller.add_component(Sink(name="K1"))
controller.connect("S1", "K1")

print("运行模拟以生成 'history' 对象...")
history_original = []
history_generator = controller.run(num_steps=10, dt=1)
for status in history_generator:
    current_step_history = {}
    for name, comp in controller.components.items():
        current_step_history[name] = {'outflow': comp.get_outflow()}
    history_original.append(current_step_history)

print("模拟完成。")

## 2. 保存 `history` 对象

我们使用`pickle`模块将`history_original`对象写入到一个二进制文件中。我们需要以二进制写模式（`'wb'`）打开文件。

In [ ]:
results_dir = '../examples/results/'
if not os.path.exists(results_dir):
    os.makedirs(results_dir)

save_path = os.path.join(results_dir, 'simulation_history.pkl')

with open(save_path, 'wb') as f:
    pickle.dump(history_original, f)

print(f"'history' 对象已成功保存到: {save_path}")

## 3. 加载 `history` 对象

现在，我们可以假装关闭了程序，然后重新打开。我们从刚刚保存的文件中加载数据。这次我们需要以二进制读模式（`'rb'`）打开文件。

In [ ]:
with open(save_path, 'rb') as f:
    history_loaded = pickle.load(f)

print("从文件中成功加载 'history' 对象。")

## 4. 验证加载的数据

最后，我们使用加载回来的`history_loaded`对象来绘图，以验证其内容是否与原始对象完全相同。

In [ ]:
outflow_original = [h['K1']['outflow'] for h in history_original]
outflow_loaded = [h['K1']['outflow'] for h in history_loaded]

# 验证数据是否一致
assert np.allclose(outflow_original, outflow_loaded), "加载的数据与原始数据不符!"
print("数据一致性验证通过。")

# 绘图
plt.figure(figsize=(10, 5))
plt.plot(outflow_loaded, 'r-s', label='Loaded Data')
plt.title('Plot from Loaded Simulation History')
plt.xlabel('Time Step')
plt.ylabel('Outflow')
plt.legend()
plt.grid(True)
plt.show()

print("绘图成功，证明加载的数据是完整且可用的。")